In [ ]:
import pandas as pd
from pymc_toolkit.pymc_model import PymcModel

data = pd.read_csv('data/Poseidon.csv')

target_name = 'total_revenue'

First, we are going to find the most correlated variables versus target variable ´total_revenue´. We left the last time point to drop and incomplete week

In [ ]:
pd.set_option('display.max_rows', None)

pd.DataFrame(data.iloc[:-1,1:].corr()[target_name]).sort_values(target_name)

As we can see, we have the total_revenue, also shopify and amazon revenue, so we will thest if the total is the some of both variables.

In [ ]:
data['amazon_shopify_revenue'] = data['amazon_revenue'] + data['shopify_revenue']
data[['amazon_shopify_revenue', target_name]].plot()

Indeed, so we will repeat the test of correlation with both variables and we will considerate the most correlated variables.

In [ ]:
pd.DataFrame(data.iloc[:-1,1:].corr()['amazon_revenue']).sort_values('amazon_revenue')

We just are going to considerate the most correlated of "costs" type variable.

In [ ]:
channel_amazon = ['amazon_ads_spend','google_ads_search_brand_costs', 'google_ads_others_costs']

Repeat con shopify.

In [ ]:
pd.DataFrame(data.iloc[:-1,1:].corr()['shopify_revenue']).sort_values('shopify_revenue')

The shopify variables.

In [ ]:
channel_shopify = ['fb_bof_spend', 'google_ads_search_brand_costs', 'fb_tof_spend', 'google_ads_others_costs']

Now we have the channels variables, and we will considerate the same control variables for benchmark model.

In [ ]:
control_names = ['discounts', 'emails_us_total_emails_sent', 'influencers_us_total_reach']
channel_names = list(set(channel_amazon + channel_shopify))
data = data[['total_revenue', 'date_week'] + control_names + channel_names].copy()

model = PymcModel(client_data=data,
             target_name=target_name,
             date_column='date_week',
             channel_names=channel_names,
             control_names=control_names,
             lag_max=1,
             scale_data=True,
             saturation='michaelis_menten',
             time_varying_media=True)

In [ ]:
pfleet = model.production_fleet(n_test=12, chains=4, draws=1000,tune=500,cores=4)
pfleet.summary()

In [ ]:
pfleet.generate_report(output_html="toy_prod_fleet_benchmark.html")

Default Model – Baseline / More Complex Specification

The default model shows moderate explanatory power with an out-of-sample around 0.50 and relatively high MAPE (≈32%) on test data. While in-sample fit is strong, the gap between training and test performance suggests overfitting or excessive model complexity.

From a Bayesian diagnostics perspective, the model is technically sound: no divergences and good effective sample sizes. However, the large number of channels and parameters increases uncertainty in attribution and makes the model less robust for forecasting and budget optimization.

Summary:
Good internal fit
Weaker generalization
Higher complexity and attribution uncertainty

New Model – Simplified / Optimized Specification

The second model demonstrates substantially better generalization, with much lower MAPE (≈12%). Both training and test errors are aligned, indicating a better bias–variance trade-off.

MCMC diagnostics are excellent: no divergences, lower tree depth, strong effective sample sizes, and stable posterior estimates. The reduced parameter space leads to clearer channel effects, more reliable saturation and adstock estimates, and improved interpretability for decision-making.

Summary:

Strong predictive accuracy
Robust out-of-sample performance
Clearer, more actionable media insights

Overall Conclusion

Default Model is useful for exploratory analysis, New Model is clearly superior for production use, budget allocation, and strategic decision-making. It delivers higher predictive accuracy, stronger stability, and better interpretability, making it the preferred MMM for optimization and scenario planning.